# raster-genie 01 — Smokehouse Creek: from COGs to a Genie answer

**One event, end to end.** In February 2024 the Smokehouse Creek fire burned more than a million acres of the Texas Panhandle. This notebook answers the three questions an insurer or emergency manager asks first — *how much land burned, how severe was it, and how many people live there* — from raster data, with SQL, and then asks the same question of a Genie Agent:

- **Land** a perimeter, the counties it touches, Sentinel-2 scenes from before and after the fire, and WorldPop population.
- **Stack** each Sentinel-2 scene into one multi-band master COG, and register every COG in a `rasters` table through a **FILE** column.
- **Query** burn severity in plain SQL: `gbx_rst_fromfile` reads only the COG blocks the perimeter needs, `gbx_rst_nbr` and `gbx_rst_mapalgebra` build the burn index, `gbx_rst_histogram` counts the severity classes.
- **Ask** the Genie Agent the same question and compare.

**Result:** acres burned by severity class, the people inside the perimeter, a severity map, and the Genie Agent's own answer.

---
_Tables in `geospatial_catalog.raster_genie`: `aois`, `rasters`, `coverage` (view)._

In [ ]:
%run ./config_nb

In [ ]:
FORCE_REBUILD = False

## 1. Why FILE + COG

A **FILE** column holds a reference to each master COG on the Volume: the table is the catalog, nothing is copied, and Unity Catalog opens the file under the caller's grants. A **Cloud Optimized GeoTIFF** keeps its pixels in 512-pixel blocks with overviews, so `gbx_rst_fromfile` fetches only the bytes a question needs: a query over one fire reads the blocks that fire covers, not the whole file.

## 2. The fire perimeter and the counties it touches

`FirePerimeterDownloader` pulls the incident's latest perimeter from the NIFC WFIGS service (EPSG:4326 GeoJSON). The TIGER cartographic boundary files give the Texas counties; we keep the ones the perimeter touches, plus the Texas outline. Together they are the `aois` table: the named areas every raster is joined to.

In [ ]:
import json

import requests
from shapely import wkb

fire_dl = fire_perimeters.download(FIRE_NAME, FIRE_DIR, year=FIRE_YEAR, state=FIRE_STATE, spark=spark)
if not fire_dl.first()["is_out_file_valid"]:
    raise RuntimeError(f"No perimeter found for {FIRE_NAME} ({FIRE_STATE}, {FIRE_YEAR})")
fire_geoms = fire_perimeters.read(FIRE_DIR, FIRE_NAME, FIRE_YEAR, state=FIRE_STATE, spark=spark)

# The perimeter bbox, padded: it frames every Sentinel-2 and WorldPop download below.
_b = [wkb.loads(bytes(r["geom_0"])).bounds for r in fire_geoms.select("geom_0").collect()]
FIRE_BBOX = (min(x[0] for x in _b), min(x[1] for x in _b), max(x[2] for x in _b), max(x[3] for x in _b))
S2_BBOX = (
    FIRE_BBOX[0] - BBOX_PAD, FIRE_BBOX[1] - BBOX_PAD,
    FIRE_BBOX[2] + BBOX_PAD, FIRE_BBOX[3] + BBOX_PAD,
)
print(f"... perimeter bbox {tuple(round(v, 3) for v in FIRE_BBOX)}")
print(f"... download bbox  {tuple(round(v, 3) for v in S2_BBOX)} (+{BBOX_PAD} deg)")

In [ ]:
TIGER = "https://www2.census.gov/geo/tiger/GENZ2024/shp"
for _name in ("cb_2024_us_county_500k.zip", "cb_2024_us_state_500k.zip"):
    _dst = f"{CONTEXT_DIR}/{_name}"
    if FORCE_REBUILD or not os.path.exists(_dst):
        _resp = requests.get(f"{TIGER}/{_name}", timeout=180, stream=True)
        _resp.raise_for_status()
        with open(_dst, "wb") as fh:
            for chunk in _resp.iter_content(chunk_size=1 << 20):
                fh.write(chunk)
        print(f"... TIGER {_name} -> {_dst}")

AOI_COLS = ["aoi_id", "name", "kind", "geom", "description"]

fire = fire_geoms.select(
    F.lit(FIRE_AOI_ID).alias("aoi_id"),
    F.lit("Smokehouse Creek fire").alias("name"),
    F.lit("fire").alias("kind"),
    F.expr("st_setsrid(st_geomfromwkb(geom_0), 4326)").alias("geom"),
    F.lit("Latest NIFC WFIGS perimeter of the 2024 Smokehouse Creek fire").alias("description"),
)

tx_counties = (
    spark.read.format("shapefile_gbx").load(f"{CONTEXT_DIR}/cb_2024_us_county_500k.zip")
    .filter(F.col("STATEFP") == "48")
    .select(
        F.concat(F.lit("county_"), F.col("GEOID")).alias("aoi_id"),
        F.col("NAME").alias("name"),
        F.lit("county").alias("kind"),
        F.expr("st_setsrid(st_geomfromwkb(geom_0), 4326)").alias("geom"),
        F.concat(F.col("NAME"), F.lit(" County, Texas")).alias("description"),
    )
)
# Keep the counties the perimeter touches.
touched = (
    tx_counties.alias("c")
    .join(fire.select(F.col("geom").alias("fire_geom")).alias("f"), F.expr("st_intersects(c.geom, f.fire_geom)"))
    .select(*[F.col(f"c.{c}") for c in AOI_COLS])
)
texas = (
    spark.read.format("shapefile_gbx").load(f"{CONTEXT_DIR}/cb_2024_us_state_500k.zip")
    .filter(F.col("STATEFP") == "48")
    .select(
        F.lit("tx").alias("aoi_id"),
        F.lit("Texas").alias("name"),
        F.lit("state").alias("kind"),
        F.expr("st_setsrid(st_geomfromwkb(geom_0), 4326)").alias("geom"),
        F.lit("State of Texas, TIGER cartographic boundary").alias("description"),
    )
)

aois = finalize_delta(fire.select(*AOI_COLS).unionByName(touched).unionByName(texas), "aois", do_display=False)
aois.select("aoi_id", "name", "kind", "description").orderBy("kind", "name").display()

## 3. Sentinel-2: the scenes, windowed and stacked

`StacClient.search` queries Microsoft Planetary Computer's `sentinel-2-l2a` collection over the padded perimeter bbox for two windows, before and after the fire. For each window we keep the **least-cloud scene per MGRS tile** (cloud cover ≤ `MAX_CLOUD`, a scene-level figure), so the fire is covered by one clean pre scene and one clean post scene per tile. `StacClient.download` then fetches the seven bands we need (B02, B03, B04, B08, B11, B12 and the scene classification SCL) **windowed to the padded bbox** — only that part of each scene crosses the network.

In [ ]:
from pyspark.sql.window import Window

SCENE_ASSETS = "s2_scene_assets"
S2_BAND_NAMES = ["B02", "B03", "B04", "B08", "B11", "B12", "SCL"]

# The search area: the padded bbox as a GeoJSON polygon.
_x0, _y0, _x1, _y1 = S2_BBOX
aoi_polygon = spark.createDataFrame(
    [(json.dumps({"type": "Polygon", "coordinates": [[[_x0, _y0], [_x1, _y0], [_x1, _y1], [_x0, _y1], [_x0, _y0]]]}),)],
    ["geojson"],
)


def least_cloud_assets(window, phase):
    """One window's search: the least-cloud scene per MGRS tile, with the hrefs of its bands."""
    found = stac_client.search(aoi_polygon, geojson_col="geojson", collections=["sentinel-2-l2a"], datetime=window)
    scenes = (
        found.dropDuplicates(["item_id"])
        .select(
            "item_id",
            "date",
            F.col("item_properties")["eo:cloud_cover"].cast("double").alias("cloud_pct"),
            F.col("item_properties")["s2:processing_baseline"].alias("baseline"),
            F.regexp_extract("item_id", r"_T(\d{2}[A-Z]{3})_", 1).alias("mgrs_tile"),
        )
        .filter(F.col("cloud_pct") <= MAX_CLOUD)
    )
    rank = Window.partitionBy("mgrs_tile").orderBy("cloud_pct", "item_id")
    best = scenes.withColumn("rn", F.row_number().over(rank)).filter("rn = 1").drop("rn")
    return (
        found.filter(F.col("asset_name").isin(*S2_BAND_NAMES))
        .select("item_id", "asset_name", "href")
        .join(best, "item_id")
        .withColumn("phase", F.lit(phase))
    )


if FORCE_REBUILD or not spark.catalog.tableExists(SCENE_ASSETS):
    chosen = least_cloud_assets(PRE_WINDOW, "pre").unionByName(least_cloud_assets(POST_WINDOW, "post"))
    finalize_delta(chosen, SCENE_ASSETS, do_display=False)
else:
    print(f"... {SCENE_ASSETS} exists; skipping the STAC search (FORCE_REBUILD=False)")

(
    spark.table(SCENE_ASSETS)
    .select("phase", "mgrs_tile", "item_id", "date", "cloud_pct", "baseline")
    .distinct()
    .orderBy("phase", "mgrs_tile")
    .display()
)

In [ ]:
# Download the chosen bands, windowed to the padded bbox. Files that are already valid on the
# Volume are skipped, so a re-run downloads nothing new.
downloads = stac_client.download(
    spark.table(SCENE_ASSETS).select("item_id", "asset_name", "href"),
    S2_RAW_DIR,
    bbox=list(S2_BBOX),
    bbox_crs="EPSG:4326",
).select("item_id", "asset_name", "out_file_path", "out_file_sz", "is_out_file_valid").collect()

failed = [r for r in downloads if not r["is_out_file_valid"]]
print(f"... {len(downloads) - len(failed)} of {len(downloads)} band files valid in {S2_RAW_DIR}")
if failed:
    raise RuntimeError(f"{len(failed)} band downloads failed, e.g. {failed[0]['item_id']} {failed[0]['asset_name']}")

Now stack each scene. `stack_scene` (in `lakeflow/land/scenes.py`) writes **one master COG per scene**: the seven bands on the 10 m grid of B04 (the 20 m bands B11, B12 and SCL are resampled, bilinear for reflectance and nearest for SCL), 512-pixel blocks, overviews at 2/4/8/16, GeoBrix's own compression options, nodata 0, and the band names as band descriptions. Reflectance has Sentinel-2's +1000 processing offset removed first, so the indices are taken on true reflectance. The file carries no internal mask band: nodata is the only validity signal, which is what the Unity Catalog windowed reader plans for.

In [ ]:
import datetime
import sys

import rasterio

# stack_scene lives in lakeflow/land/scenes.py, beside this notebook.
_lakeflow = os.path.join(os.getcwd(), "lakeflow")
if not os.path.exists(os.path.join(_lakeflow, "land", "scenes.py")):
    raise FileNotFoundError(
        f"lakeflow/land/scenes.py was not found next to this notebook ({os.getcwd()}). "
        "Fix: open this series from a Git folder, or copy the lakeflow/ folder next to the notebooks."
    )
sys.path.insert(0, _lakeflow)
from land.scenes import S2_BANDS, boa_offset_for, stack_scene

band_files = {}
for r in downloads:
    band_files.setdefault(r["item_id"], {})[r["asset_name"]] = r["out_file_path"]

scene_rows = (
    spark.table(SCENE_ASSETS)
    .select("item_id", "phase", "mgrs_tile", "date", "cloud_pct", "baseline")
    .distinct()
    .orderBy("phase", "mgrs_tile")
    .collect()
)

masters = []  # one record per scene master COG
for s in scene_rows:
    day = s["date"][:10]
    asset_id = f"s2_{s['mgrs_tile']}_{day.replace('-', '')}"
    out = f"{S2_MASTER_DIR}/{asset_id}.tif"
    if FORCE_REBUILD or not os.path.exists(out):
        stack_scene(band_files[s["item_id"]], out, bands=S2_BANDS, ref="B04", boa_offset=boa_offset_for(s["baseline"]))
        print(f"... stacked {asset_id} ({s['phase']}, cloud {s['cloud_pct']:.1f}%)")
    else:
        print(f"... {asset_id} exists (skip; FORCE_REBUILD=False)")
    with rasterio.open(out) as ds:
        km2 = ds.width * ds.height * abs(ds.res[0] * ds.res[1]) / 1e6
    masters.append(
        dict(asset_id=asset_id, day=datetime.date.fromisoformat(day), cloud_pct=float(s["cloud_pct"]),
             path=out, bytes=os.path.getsize(out), km2=km2)
    )
print(f"... {len(masters)} Sentinel-2 master COGs in {S2_MASTER_DIR}")

## 4. WorldPop population

WorldPop's 2020 constrained population grid for the USA is a single national GeoTIFF. `WorldPopDownloader` windows it to the same padded bbox; where the server cannot serve byte ranges it fetches the national file once into `POP_CACHE_DIR` first, and every later call windows that local copy. `prepare_cogs` rewrites the window as a master COG in `population/`: the same layout as the Sentinel-2 scenes, so one table catalogs both.

In [ ]:
pop_dl = worldpop.download(S2_BBOX, POP_RAW_DIR, spark=spark, cache_dir=POP_CACHE_DIR).first()
if not pop_dl["is_out_file_valid"]:
    raise RuntimeError("WorldPop window does not overlap the download bbox")
pop_summary = prepare_cogs(
    pop_dl["out_file_path"], POP_DIR, blocksize=512, compress="auto", bigtiff="IF_SAFER", skip_if_exists=not FORCE_REBUILD
)
pop_cog = pop_summary["results"][0]["output_path"]
if pop_cog is None:
    raise RuntimeError(f"population COG failed: {pop_summary['results'][0]['status']}")
print(f"... population master COG: {pop_cog} ({os.path.getsize(pop_cog) / 1e6:,.1f} MB)")

## 5. Land the COGs as a FILE table

One row per master COG: its product, tier, area, date, band order, source and licence, and the **FILE EXTERNAL** column `file` that points at the COG on the Volume. All of these are marked **enhanced**, the tier for focus areas and events. We read the technical metadata from the COG itself with SQL before writing: a header-only tile gives the CRS, pixel size and bounding box (`gbx_rst_srid`, `gbx_rst_pixelheight`, `gbx_rst_boundingbox`, reprojected to a `GEOMETRY(4326)` footprint with `st_transform`), and `valid_pct` is the share of data pixels read from a coarse overview, so landing never decodes a full scene.

In [ ]:
RASTERS = f"{catalog_name}.{schema_name}.rasters"

if FORCE_REBUILD or not spark.catalog.tableExists(RASTERS):
    s2_source = "Sentinel-2 L2A, Microsoft Planetary Computer"
    s2_license = "Copernicus Sentinel data: free, full and open"
    rows = [
        (m["asset_id"], "s2_l2a", "enhanced", FIRE_AOI_ID, m["day"], list(S2_BANDS),
         s2_source, s2_license, m["cloud_pct"], m["path"], 160.0)
        for m in masters
    ]
    rows.append(
        ("pop_worldpop_2020_" + FIRE_AOI_ID, "population", "enhanced", FIRE_AOI_ID, datetime.date(2020, 1, 1),
         ["population"], "WorldPop 2020 constrained population, USA (people per pixel)", "CC BY 4.0",
         None, pop_cog, None)
    )
    landing_schema = StructType([
        StructField("asset_id", StringType()), StructField("product", StringType()),
        StructField("tier", StringType()), StructField("aoi_id", StringType()),
        StructField("acquired_date", DateType()), StructField("bands", ArrayType(StringType())),
        StructField("source", StringType()), StructField("license", StringType()),
        StructField("cloud_pct", DoubleType()), StructField("path", StringType()),
        StructField("preview_res", DoubleType()),  # pixel size of the coarse overview read for valid_pct
    ])
    landing = spark.createDataFrame(rows, landing_schema)

    # t: a header-only tile (CRS, pixel size, bounds). preview: a coarse overview read (valid pixels).
    (
        landing
        .withColumn("t", rx.rst_fromfile(F.col("path")))
        .withColumn("preview", F.expr("gbx_rst_fromfile(path, 'GTiff', NULL, NULL, preview_res)"))
        .createOrReplaceTempView("landing_tiles")
    )
    rasters_df = spark.sql("""
        SELECT asset_id, product, tier, aoi_id, acquired_date, bands,
               concat('EPSG:', CAST(gbx_rst_srid(t) AS STRING)) AS crs,
               CASE WHEN gbx_rst_srid(t) = 4326
                    THEN abs(gbx_rst_pixelheight(t)) * 111320.0
                    ELSE abs(gbx_rst_pixelwidth(t)) END AS resolution_m,
               st_transform(st_setsrid(st_geomfromwkb(gbx_rst_boundingbox(t)), gbx_rst_srid(t)), 4326) AS footprint,
               round(100.0 * gbx_rst_pixelcount(preview)[0] / (gbx_rst_width(preview) * gbx_rst_height(preview)), 1) AS valid_pct,
               cloud_pct, source, license, path
        FROM landing_tiles
    """)
    gbx_file_write(rasters_df, RASTERS, file_mode="external", file_col="file", overwrite=True)
else:
    print(f"... {RASTERS} exists (skip; FORCE_REBUILD=False)")

spark.table(RASTERS).select(
    "asset_id", "product", "tier", "acquired_date", "bands", "crs", "resolution_m", "valid_pct", "cloud_pct", "file"
).orderBy("product", "acquired_date").display()

In [ ]:
# Genie reads the comments: describe the tables and their columns.
def comment_table(tbl, table_comment, columns):
    spark.sql(f"COMMENT ON TABLE {tbl} IS '{table_comment}'")
    for col, text in columns.items():
        spark.sql(f"ALTER TABLE {tbl} ALTER COLUMN {col} COMMENT '{text}'")


comment_table(RASTERS, "One row per master COG: Sentinel-2 L2A scenes and WorldPop population, each registered through a FILE column.", {
    "asset_id": "Unique id of the master COG",
    "product": "s2_l2a (Sentinel-2 L2A), dem, land_cover or population",
    "tier": "standard (all of Texas) or enhanced (focus areas and events)",
    "aoi_id": "The area (aois.aoi_id) the asset was landed for",
    "acquired_date": "Acquisition date of the scene; the reference year for population",
    "bands": "Band names in band order; the position in this array is the 1-based band index",
    "crs": "Coordinate reference system of the COG",
    "resolution_m": "Native pixel size in metres",
    "footprint": "Extent of the COG as a GEOMETRY(4326)",
    "valid_pct": "Percent of pixels that hold data (not nodata)",
    "cloud_pct": "Scene-level cloud cover in percent (Sentinel-2 only)",
    "source": "Where the data comes from",
    "license": "Licence of the data",
    "path": "Volume path of the COG",
    "file": "FILE EXTERNAL reference to the COG; pass it to gbx_rst_fromfile",
})
comment_table(f"{catalog_name}.{schema_name}.aois", "Named areas: the fire perimeter, the Texas counties it touches and the Texas outline.", {
    "aoi_id": "Unique id of the area",
    "name": "Display name",
    "kind": "fire, county or state",
    "geom": "Boundary as a GEOMETRY(4326)",
    "description": "What the area is",
})

In [ ]:
# What landing produced, and what that means at scale.
s2_gb = sum(m["bytes"] for m in masters) / 1e9
s2_km2 = sum(m["km2"] for m in masters)
n_assets = spark.table(RASTERS).count()
print(f"... {n_assets} assets in {RASTERS}: {len(masters)} Sentinel-2 scenes ({s2_gb:,.2f} GB over about {s2_km2:,.0f} km2) + 1 population COG")
print(
    f"... at this density, one Sentinel-2 pass over all of Texas (695,662 km2) is about {s2_gb / s2_km2 * 695_662 / 1e3:,.1f} TB "
    "of master COGs, and a year of quarterly passes about four times that: the same table, the same query."
)

## 6. The `coverage` view

What do we hold, where, at what resolution and how recent? `coverage` answers that per product, tier and area by joining each COG's footprint to the `aois`. A data-product owner can ask it the first question of any engagement.

In [ ]:
spark.sql("""
    CREATE OR REPLACE VIEW coverage
    COMMENT 'What raster data we hold: one row per product, tier and area, with the best resolution, date range, asset count and mean valid percent.'
    AS
    SELECT r.product, r.tier, a.aoi_id, a.name AS aoi_name, a.kind AS aoi_kind,
           min(r.resolution_m) AS best_resolution_m,
           min(r.acquired_date) AS first_date,
           max(r.acquired_date) AS last_date,
           count(*) AS assets,
           round(avg(r.valid_pct), 1) AS mean_valid_pct
    FROM rasters r
    JOIN aois a ON st_intersects(r.footprint, a.geom)
    GROUP BY r.product, r.tier, a.aoi_id, a.name, a.kind
""")
spark.table("coverage").orderBy("aoi_kind", "aoi_name", "product").display()

## 7. Burn severity in plain SQL

The classes follow the standard dNBR table (Key & Benson, 2006, *FIREMON*, USDA Forest Service RMRS-GTR-164, as published by USGS):

| dNBR | Class | Severity |
|---|---|---|
| below 0.1 | 1 | Unburned |
| 0.1 to 0.27 | 2 | Low |
| 0.27 to 0.44 | 3 | Moderate-low |
| 0.44 to 0.66 | 4 | Moderate-high |
| 0.66 and above | 5 | High |

Key & Benson also list two regrowth classes below -0.1 (enhanced regrowth, high and low); they are folded into class 1, Unburned, here.

The query is one SQL statement over the `rasters` and `aois` tables:

1. **Pair** each MGRS tile's pre and post scene whose footprint meets the perimeter (the tile is part of the `asset_id`).
2. **Read** only the perimeter from each scene with `gbx_rst_fromfile(file, 'GTiff', <perimeter>, 'EPSG:4326', <resolution>)`: the geometry clips the window and the resolution picks the overview, so only the COG blocks the perimeter touches are fetched.
3. **Index** each read with `gbx_rst_nbr(tile, nir, swir)`, the normalized burn ratio `(NIR - SWIR) / (NIR + SWIR)`. The band numbers are looked up from the `bands` column: NIR is B08 and SWIR is B12.
4. **Difference** them with `gbx_rst_mapalgebra`: dNBR = pre NBR - post NBR.
5. **Classify** dNBR into the five classes with a second `gbx_rst_mapalgebra`, and **merge** the tiles with `gbx_rst_merge`, so the overlap between neighbouring MGRS tiles is counted once.
6. **Count** the pixels per class with `gbx_rst_histogram`, and turn pixels into area with the tile's own pixel size. `pct_of_valid_pixels` is each class's share of the pixels that hold data inside the perimeter.

Scene-level cloud cover is at most `MAX_CLOUD` percent and this slice does not mask clouds pixel by pixel; the SCL band is in each COG for that. The answer is saved as `burn_severity_summary`, so a re-run does not read the COGs again.

The window one `gbx_rst_fromfile` call returns must fit the **64 MiB** a Unity Catalog function can return. Before the query we size it: the perimeter's bounding box in the scene CRS, divided into pixels at the read resolution, times the seven bands at two bytes each. If that is over 64 MiB the read steps up to the next coarser overview of the COG (20, 40, 80, then 160 m), and the cell says which one it chose.

In [ ]:
import math

from pyproj import Transformer

FROMFILE_CAP_BYTES = 64 * 1024**2
NATIVE_M = 10.0
LEVELS_M = [NATIVE_M * f for f in (1, 2, 4, 8, 16)]  # pixel sizes of the master COG and its overviews

scene_crs = spark.table(RASTERS).filter("product = 's2_l2a'").select("crs").first()["crs"]
_to_scene = Transformer.from_crs("EPSG:4326", scene_crs, always_xy=True)
_xy = [_to_scene.transform(x, y) for x in (FIRE_BBOX[0], FIRE_BBOX[2]) for y in (FIRE_BBOX[1], FIRE_BBOX[3])]
extent_x = max(p[0] for p in _xy) - min(p[0] for p in _xy)
extent_y = max(p[1] for p in _xy) - min(p[1] for p in _xy)


def window_bytes(res_m):
    """Decoded size of the perimeter's window at a pixel size: pixels x bands x 2 bytes."""
    return math.ceil(extent_x / res_m) * math.ceil(extent_y / res_m) * len(S2_BANDS) * 2


# gbx_rst_fromfile reads the coarsest overview at least as fine as the resolution asked for.
burn_res_m = max([m for m in LEVELS_M if m <= BURN_RES_M] or [NATIVE_M])
print(f"... asked for {BURN_RES_M:g} m: the read uses the {burn_res_m:g} m overview, "
      f"a window of {window_bytes(burn_res_m) / 1024**2:,.1f} MiB (limit {FROMFILE_CAP_BYTES / 1024**2:.0f} MiB)")
while window_bytes(burn_res_m) > FROMFILE_CAP_BYTES:
    coarser = [m for m in LEVELS_M if m > burn_res_m]
    if not coarser:
        raise RuntimeError("The perimeter window is over 64 MiB at every overview. Fix: query a smaller part of the perimeter.")
    print(f"... {burn_res_m:g} m is over the limit, so the read steps up to the {coarser[0]:g} m overview")
    burn_res_m = coarser[0]
print(f"... burn severity is read at {burn_res_m:g} m ({window_bytes(burn_res_m) / 1024**2:,.1f} MiB per window)")
map_res_m = max(MAP_RES_M, burn_res_m)

In [ ]:
BURN_CTES = """
WITH perimeter AS (
  SELECT geom FROM aois WHERE aoi_id = '{fire_id}'
),
scenes AS (
  SELECT asset_id, regexp_extract(asset_id, '^s2_([0-9A-Z]+)_', 1) AS mgrs_tile,
         acquired_date, bands, file, footprint
  FROM rasters
  WHERE product = 's2_l2a' AND aoi_id = '{fire_id}'
),
pairs AS (
  SELECT pre.mgrs_tile, pre.bands, pre.file AS pre_file, post.file AS post_file, p.geom
  FROM scenes pre
  JOIN scenes post ON post.mgrs_tile = pre.mgrs_tile
  CROSS JOIN perimeter p
  WHERE pre.acquired_date  BETWEEN DATE'{pre_from}'  AND DATE'{pre_to}'
    AND post.acquired_date BETWEEN DATE'{post_from}' AND DATE'{post_to}'
    AND st_intersects(pre.footprint, p.geom)
),
reads AS (
  SELECT mgrs_tile, bands,
         gbx_rst_fromfile(pre_file,  'GTiff', geom, 'EPSG:4326', {res}) AS pre_tile,
         gbx_rst_fromfile(post_file, 'GTiff', geom, 'EPSG:4326', {res}) AS post_tile
  FROM pairs
),
dnbr AS (
  SELECT mgrs_tile,
         gbx_rst_mapalgebra(array(
           gbx_rst_nbr(pre_tile,  CAST(array_position(bands, 'B08') AS INT), CAST(array_position(bands, 'B12') AS INT)),
           gbx_rst_nbr(post_tile, CAST(array_position(bands, 'B08') AS INT), CAST(array_position(bands, 'B12') AS INT))
         ), 'A - B') AS tile
  FROM reads
  WHERE pre_tile IS NOT NULL AND post_tile IS NOT NULL
),
severity AS (
  SELECT gbx_rst_merge(collect_list(
           gbx_rst_mapalgebra(array(tile), 'A < 0.1 ? 1 : (A < 0.27 ? 2 : (A < 0.44 ? 3 : (A < 0.66 ? 4 : 5)))')
         )) AS tile
  FROM dnbr
)
"""

BURN_AREA = """
SELECT pos + 1 AS class_id,
       CASE pos + 1 WHEN 1 THEN 'Unburned' WHEN 2 THEN 'Low' WHEN 3 THEN 'Moderate-low'
                    WHEN 4 THEN 'Moderate-high' ELSE 'High' END AS severity,
       n AS pixels,
       round(n * abs(gbx_rst_pixelwidth(tile) * gbx_rst_pixelheight(tile)) / 1e6, 1) AS km2,
       round(n * abs(gbx_rst_pixelwidth(tile) * gbx_rst_pixelheight(tile)) / 4046.8564224, 0) AS acres,
       round(100.0 * n / sum(n) OVER (), 1) AS pct_of_valid_pixels
FROM (SELECT tile, gbx_rst_histogram(tile, 5, 0.5, 5.5, false)['band_1'] AS counts FROM severity)
LATERAL VIEW posexplode(counts) h AS pos, n
ORDER BY class_id
"""

burn_params = dict(
    fire_id=FIRE_AOI_ID,
    pre_from=PRE_WINDOW.split("/")[0], pre_to=PRE_WINDOW.split("/")[1],
    post_from=POST_WINDOW.split("/")[0], post_to=POST_WINDOW.split("/")[1],
)
# The query reads the COGs, so run it once and keep the answer.
burn_area = finalize_delta(
    spark.sql(BURN_CTES.format(res=burn_res_m, **burn_params) + BURN_AREA), "burn_severity_summary", do_display=False
)
if burn_area.count() == 0:
    raise RuntimeError("No pre and post scene pair of one MGRS tile meets the perimeter; check the windows and MAX_CLOUD")
burn_area.display()

Now the people. The WorldPop COG is read the same way, clipped to the perimeter, and the population inside is the pixel mean times the count of valid pixels.

In [ ]:
people_in_perimeter = spark.sql(f"""
    SELECT round(avg_values[0] * valid_counts[0]) AS people_in_perimeter
    FROM (
      SELECT gbx_rst_avg(t) AS avg_values, gbx_rst_pixelcount(t) AS valid_counts
      FROM (
        SELECT gbx_rst_fromfile(r.file, 'GTiff', a.geom, 'EPSG:4326') AS t
        FROM rasters r CROSS JOIN aois a
        WHERE r.product = 'population' AND a.aoi_id = '{FIRE_AOI_ID}'
          AND st_intersects(r.footprint, a.geom)
      )
    )
""").first()[0]

_acres = burn_area.agg(F.sum("acres")).first()[0]
_burned = burn_area.filter("class_id >= 2").agg(F.sum("acres")).first()[0]
print(f"... {_acres:,.0f} acres inside the perimeter, {_burned:,.0f} of them at low severity or worse")
if people_in_perimeter is None:
    print("... no WorldPop pixel with data falls inside the perimeter; check the population COG and its footprint")
else:
    print(f"... about {people_in_perimeter:,.0f} people live inside the perimeter (WorldPop 2020)")

## 8. The map

The same query, read at the coarser `MAP_RES_M` (or the severity resolution, if that is coarser) so the figure stays light, gives the merged severity-class raster. `vizx.plot_tile` drapes it over an Esri basemap, with the NIFC perimeter outlined.

In [ ]:
import geopandas as gpd
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

SEVERITY_CLASSES = [
    ("Unburned", "#a6d96a"), ("Low", "#ffffbf"), ("Moderate-low", "#fdae61"),
    ("Moderate-high", "#f46d43"), ("High", "#a50026"),
]
severity_cmap = ListedColormap([color for _, color in SEVERITY_CLASSES])
# Class k always takes colour k, whichever classes the raster holds.
severity_norm = BoundaryNorm(np.arange(0.5, 6.5, 1.0), severity_cmap.N)

severity_row = spark.sql(
    BURN_CTES.format(res=map_res_m, **burn_params)
    + "SELECT tile, gbx_rst_srid(tile) AS srid FROM severity"
).first()
fire_wkt = spark.sql(f"SELECT st_astext(geom) AS wkt FROM aois WHERE aoi_id = '{FIRE_AOI_ID}'").first()["wkt"]

ax = plot_tile(
    severity_row["tile"],
    band=1,
    cmap=severity_cmap,
    basemap_source="topo",
    title="Smokehouse Creek fire: dNBR burn severity inside the NIFC perimeter",
)
next(im for im in ax.images if im.get_zorder() == max(i.get_zorder() for i in ax.images)).set_norm(severity_norm)
ax.legend(
    handles=[Patch(facecolor=color, edgecolor="grey", label=f"{k}  {name}") for k, (name, color) in enumerate(SEVERITY_CLASSES, start=1)],
    title="dNBR severity class", loc="lower left",
)
gpd.GeoSeries.from_wkt([fire_wkt], crs=4326).to_crs(int(severity_row["srid"])).boundary.plot(
    ax=ax, color="black", linewidth=1.2
)

## 9. Ask the Genie Agent

The SQL above is what a person writes when they know the functions. A **Genie Agent** is given the same tables, the `gbx_rst_*` functions and a short set of instructions, and composes the query itself from a plain-English question. The call below uses Agent mode and shows the SQL the agent wrote next to its answer, so the two approaches can be compared. It needs the id of the raster-genie Genie space in `GENIE_SPACE_ID` (set in `config_nb`). The SQL above counts the people inside the whole fire perimeter, burned or not, so compare the agent's population figure on that basis.

In [ ]:
Q3 = "How much land burned in the Smokehouse Creek fire, how severe, and how many people live in the burned area?"

if not GENIE_SPACE_ID:
    print("... GENIE_SPACE_ID is empty in config_nb; skipping the Genie Agent step.")
    print("... Set it to the id of the raster-genie Genie space and run this cell again.")
else:
    answer = ask_genie(GENIE_SPACE_ID, Q3)
    print(f"Question: {Q3}")
    print(f"Status:   {answer['status']} in {answer['seconds']} s")
    if answer["error"]:
        print(f"Error:    {answer['error']}")
    for i, statement in enumerate(answer["sql"], start=1):
        print(f"\n-- SQL the agent ran ({i} of {len(answer['sql'])})\n{statement}")
    print("\nAnswer:")
    for text in answer["text"]:
        print(text)

## What we built

- **`aois`** (Delta): the Smokehouse Creek perimeter, the Texas counties it touches and the Texas outline.
- **`rasters`** (Delta, FILE EXTERNAL): one row per master COG, with its footprint, CRS, resolution, valid percent, band order, source and licence.
- **`coverage`** (view): what we hold per product, tier and area.
- **`burn_severity_summary`** (Delta): the severity query's answer, kept so the notebook does not read the COGs twice.
- The **burn severity** by class in acres and square kilometres, the **people** inside the perimeter, and a severity map.
- The Genie Agent's SQL and answer to the same question.

GeoBrix: `StacClient`, `FirePerimeterDownloader`, `WorldPopDownloader`, `prepare_cogs`, `gbx_file_write`, `gbx_rst_fromfile`, `gbx_rst_nbr`, `gbx_rst_mapalgebra`, `gbx_rst_merge`, `gbx_rst_histogram`, `gbx_rst_avg`, `gbx_rst_pixelcount`, `genie.ask`, `vizx.plot_tile`.